# HMI Solar Data Processing Pipeline
Processes SDO/HMI magnetogram data from the SDO-ML Zarr archive alongside SPoCA coronal hole maps, computing global and grid-based intensity metrics at 6-hour cadence.

In [ ]:
import os
import gc
import requests
import numpy as np
import pandas as pd
import zarr
import s3fs
import astropy.units as u
from astropy.io import fits
from sunpy.coordinates import frames, SphericalScreen
import sunpy.map
import reproject
from tqdm import tqdm
import warnings
from sunpy.util.exceptions import SunpyMetadataWarning

warnings.filterwarnings('ignore', category=SunpyMetadataWarning)

## 1. Connect to SDO-ML Zarr Archive (S3)

In [ ]:
print("Connecting to SDO-ML HMI 2010 Zarr...")

s3 = s3fs.S3FileSystem(anon=True)
AWS_ZARR_2010 = "s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2_hmi_small.zarr/2010"
store = s3fs.S3Map(root=AWS_ZARR_2010, s3=s3)
root = zarr.open(store=store, mode="r")

data_211 = root["Bz"]
zarr_times = pd.to_datetime(list(data_211.attrs['DATE-OBS']))

print(f"Zarr dataset shape: {data_211.shape}")
print(f"Available time range: {zarr_times[0]} → {zarr_times[-1]}")

## 2. Define Target Timestamps and Grid Bounds

In [ ]:
# 6-hour cadence: 00:00, 06:00, 12:00, 18:00 UTC every day
# Range is bounded by actual data availability, not arbitrary:
#   - sdomlv2_hmi_small.zarr only has a "2010" year folder (no 2011+ data exists)
#   - SPoCA rob_spoca_ch CH maps don't start until 2010-05-13
target_times = pd.date_range(
    start='2010-05-13 00:00:00',
    end='2010-12-31 18:00:00',
    freq='6H',
    tz='UTC'
)

# Grid definition bounds
abs_lat_bounds = [0, 45, 90]
lon_bounds = [-90, -30, 30, 90]

print(f"Total target timestamps: {len(target_times)}")
print(f"Date range: {target_times[0]} → {target_times[-1]}")

## 3. Main Processing Loop

In [ ]:
def compute_global_stats(pix):
    """Return (energy, entropy, variance) for an array of magnetogram pixel values."""
    energy = np.nanmean(pix ** 2)
    counts, _ = np.histogram(pix, bins=100)
    probs = counts / counts.sum()
    probs = probs[probs > 0]  # exclude empty bins before log
    entropy = -np.sum(probs * np.log(probs))
    variance = np.nanvar(pix)
    return energy, entropy, variance


compiled_results = []

print(f"Starting processing of {len(target_times)} instances...")

for target_time in tqdm(target_times, desc="Processing 2010"):

    row_data = {'timestamp': target_time}

    # --- A. Find closest Zarr frame ---
    idx = np.argmin(np.abs(zarr_times - target_time))
    obs_time = zarr_times[idx]

    # Skip if closest image is more than 2 hours away
    if abs((obs_time - target_time).total_seconds()) > 7200:
        print(f"Warning: Closest image for {target_time} is {obs_time}. Skipping.")
        continue

    # Build SunPy Map from Zarr metadata
    header = {
        'CDELT1': data_211.attrs['CDELT1'][idx], 'CDELT2': data_211.attrs['CDELT2'][idx],
        'CRPIX1': data_211.attrs['CRPIX1'][idx], 'CRPIX2': data_211.attrs['CRPIX2'][idx],
        'CRVAL1': data_211.attrs['CRVAL1'][idx], 'CRVAL2': data_211.attrs['CRVAL2'][idx],
        'CTYPE1': 'HPLN-TAN', 'CTYPE2': 'HPLT-TAN',
        'CUNIT1': 'arcsec', 'CUNIT2': 'arcsec',
        'DATE-OBS': obs_time.isoformat(),
        'RSUN_OBS': data_211.attrs['RSUN_OBS'][idx],
        'NAXIS1': data_211.shape[2], 'NAXIS2': data_211.shape[1]
    }
    img_data = data_211[idx]
    sdoml_map = sunpy.map.Map(img_data, header)

    # --- B. Download SPoCA Coronal Hole FITS ---
    # NOTE: Adjust URL path if files are nested in year/month subdirectories
    time_str = target_time.strftime("%Y%m%d_%H%M%S")
    fits_url = f"https://spoca.oma.be/spoca4tap/rob_spoca_ch/ch_map/{time_str}.ch_map.fits"
    temp_fits_file = f"temp_{time_str}.fits"

    try:
        response = requests.get(fits_url, timeout=15)
        if response.status_code == 200:
            with open(temp_fits_file, 'wb') as f:
                f.write(response.content)
        else:
            continue  # File missing on server

        with fits.open(temp_fits_file) as hdu_list:
            ch_data = hdu_list[1].data
            ch_header = hdu_list[1].header

        ch_map_full_res = sunpy.map.Map(ch_data, ch_header)
        ch_data_reprojected, _ = reproject.reproject_interp(
            ch_map_full_res, sdoml_map.wcs, shape_out=sdoml_map.data.shape
        )
        binary_mask = (ch_data_reprojected > 0.5).astype(np.uint8)

    except Exception as e:
        print(f"Failed to process {fits_url}: {e}")
        if os.path.exists(temp_fits_file):
            os.remove(temp_fits_file)
        continue
    finally:
        if os.path.exists(temp_fits_file):
            os.remove(temp_fits_file)

    # --- C. Global Coronal Hole Features ---
    hpc_coords = sunpy.map.all_coordinates_from_map(sdoml_map)

    # Keep both signs: negative flux is required for R2
    ch_pix = img_data[binary_mask.astype(bool)]
    ch_pix = ch_pix[~np.isnan(ch_pix)]

    if len(ch_pix) > 0:
        # R2 — magnetic flux imbalance ratio
        # R2 = 2 * |0.5 - Phi+ / (Phi+ + |Phi-|)|
        phi_pos = np.sum(ch_pix[ch_pix > 0])
        phi_neg = np.sum(ch_pix[ch_pix < 0])
        denom = phi_pos + np.abs(phi_neg)
        row_data['global_R2'] = 2 * np.abs(0.5 - phi_pos / denom) if denom > 0 else np.nan

        row_data['global_energy'], row_data['global_entropy'], row_data['global_variance'] = compute_global_stats(ch_pix)
    else:
        # No coronal hole pixels detected: fall back to the full solar disk
        r_dist = np.sqrt(hpc_coords.Tx**2 + hpc_coords.Ty**2)
        disk_mask = r_dist < (sdoml_map.rsun_obs * 0.99)
        disk_pix = img_data[disk_mask]
        disk_pix = disk_pix[~np.isnan(disk_pix)]

        row_data['global_R2'] = 1
        if len(disk_pix) > 0:
            row_data['global_energy'], row_data['global_entropy'], row_data['global_variance'] = compute_global_stats(disk_pix)
        else:
            row_data['global_energy'] = np.nan
            row_data['global_entropy'] = np.nan
            row_data['global_variance'] = np.nan

    # --- D. 6 Symmetric Grid Metrics ---
    # SphericalScreen assumes off-disk pixels lie on a sphere so the transform
    # resolves instead of returning NaN + warning for every off-disk pixel
    with SphericalScreen(sdoml_map.observer_coordinate):
        hgs_coords = hpc_coords.transform_to(frames.HeliographicStonyhurst)
    lat = hgs_coords.lat.to('deg').value
    lon = hgs_coords.lon.to('deg').value
    abs_lat = np.abs(lat)

    # TODO: Add per-grid-cell metrics here using abs_lat_bounds and lon_bounds

    compiled_results.append(row_data)

    # Memory cleanup after each heavy reprojection
    del sdoml_map, ch_map_full_res, ch_data_reprojected, binary_mask, hpc_coords, hgs_coords
    gc.collect()

print(f"Processing complete. {len(compiled_results)} records collected.")

## 4. Save Results

In [ ]:
df_results = pd.DataFrame(compiled_results)
df_results.to_csv("hmi_2010_6h.csv", index=False)
print("Saved to 'hmi_2010_6h.csv'")
df_results.head()

## 5. Quick Summary Stats

In [ ]:
print(f"Total rows: {len(df_results)}")
print(f"Date range: {df_results['timestamp'].min()} → {df_results['timestamp'].max()}")
df_results.describe()